# 🔍 Advanced Search & Indexing

This notebook demonstrates:
- Building and tuning **HNSW** indexes
- **BM25** keyword search (via `pg_textsearch`)
- **Ensemble search** (metadata + keyword + semantic)
- **Metadata operators** (`$gt`, `$in`, `$and`, `$or`, etc.)
- **Recall measurement** and **query plans**
- **Universal keyword search** across content + metadata

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from langchain_core.documents import Document

from pgvectordb import Config, DistanceMetric, pgVectorDB

In [2]:
db = pgVectorDB(
    collection_name="nb_advanced",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await db.initialize(overwrite_existing=True)
print("✅ Ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Ready


In [3]:
# Sample dataset: tech articles
docs = [
    Document(
        page_content="PostgreSQL 17 introduces incremental backup and improved COPY performance.",
        metadata={"topic": "database", "year": 2024, "author": "Alice"},
    ),
    Document(
        page_content="pgvector 0.8 adds iterative scan and binary quantization support.",
        metadata={"topic": "database", "year": 2024, "author": "Bob"},
    ),
    Document(
        page_content="DiskANN enables billion-scale vector search with memory optimization.",
        metadata={"topic": "database", "year": 2023, "author": "Alice"},
    ),
    Document(
        page_content="Transformer models like BERT revolutionized natural language understanding.",
        metadata={"topic": "AI", "year": 2019, "author": "Carol"},
    ),
    Document(
        page_content="RAG pipelines combine retrieval with LLM generation for grounded responses.",
        metadata={"topic": "AI", "year": 2024, "author": "Alice"},
    ),
    Document(
        page_content="BM25 remains the gold standard for lexical information retrieval.",
        metadata={"topic": "AI", "year": 2020, "author": "Dave"},
    ),
    Document(
        page_content="Kubernetes operators automate complex stateful application management.",
        metadata={"topic": "devops", "year": 2023, "author": "Bob"},
    ),
    Document(
        page_content="Terraform infrastructure-as-code enables reproducible cloud deployments.",
        metadata={"topic": "devops", "year": 2022, "author": "Dave"},
    ),
    Document(
        page_content="Cosine similarity measures the angle between embedding vectors.",
        metadata={"topic": "AI", "year": 2021, "author": "Carol"},
    ),
    Document(
        page_content="Connection pooling with PgBouncer improves PostgreSQL scalability.",
        metadata={"topic": "database", "year": 2023, "author": "Carol"},
    ),
]

ids = await db.add_documents(docs)
print(f"✅ Added {len(ids)} documents")

✅ Added 10 documents


## 1. Build HNSW Index & Tune Parameters

In [4]:
await db.build_index(
    metric=DistanceMetric.COSINE,
    m=16,  # max connections per node
    ef_construction=64,  # build-time candidate list
)
print("✅ HNSW index built")

# Tune query-time parameters
await db.set_query_params(ef_search=100)
print("✅ ef_search set to 100")

✅ HNSW index built
✅ ef_search set to 100


In [5]:
# Check index stats
idx_stats = await db.get_index_stats()
print("📊 Index Stats:")
for k, v in idx_stats.items():
    print(f"  {k}: {v}")

📊 Index Stats:
  index_type: hnsw
  index_built: True
  vector_size: 384
  indexes: [{'name': 'nb_advanced_pkey', 'definition': 'CREATE UNIQUE INDEX nb_advanced_pkey ON public.nb_advanced USING btree (langchain_id)'}, {'name': 'idx_nb_advanced_content_tsvector', 'definition': 'CREATE INDEX idx_nb_advanced_content_tsvector ON public.nb_advanced USING gin (content_tsvector)'}, {'name': 'idx_nb_advanced_content_trgm', 'definition': 'CREATE INDEX idx_nb_advanced_content_trgm ON public.nb_advanced USING gin (content gin_trgm_ops)'}, {'name': 'nb_advancedlangchainvectorindex', 'definition': "CREATE INDEX nb_advancedlangchainvectorindex ON public.nb_advanced USING hnsw (embedding vector_cosine_ops) WITH (m='16', ef_construction='64')"}]
  table_stats: {'inserts': 0, 'updates': 0, 'deletes': 0, 'live_tuples': 0, 'dead_tuples': 0, 'last_vacuum': None, 'last_autovacuum': None, 'last_analyze': None, 'last_autoanalyze': None, 'bloat_ratio': 0}
  size: {'total': '160 kB', 'table': '56 kB', 'indexes

## 2. Advanced Metadata Filters

pgVectorDB supports MongoDB-style operators: `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$ne`, `$exists`, `$like`, `$ilike`, `$and`, `$or`.

In [6]:
# Year > 2022 AND topic = database
results = await (
    db.query("vector indexing")
    .semantic()
    .where({"$and": [{"year": {"$gt": 2022}}, {"topic": "database"}]})
    .limit(3)
    .to_list()
)
print("🎯 year>2022 AND topic=database:")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content'][:70]}... ({r['metadata']})")

🎯 year>2022 AND topic=database:
  [0.5596] DiskANN enables billion-scale vector search with memory optimization.... ({'topic': 'database', 'year': 2023, 'author': 'Alice', 'langchain_id': '38ac11c6-a498-4f7b-8c35-384f909a0f8c'})
  [0.5962] pgvector 0.8 adds iterative scan and binary quantization support.... ({'topic': 'database', 'year': 2024, 'author': 'Bob', 'langchain_id': 'da0553cf-a550-4252-b169-348d8be355c9'})
  [0.9118] Connection pooling with PgBouncer improves PostgreSQL scalability.... ({'topic': 'database', 'year': 2023, 'author': 'Carol', 'langchain_id': '5c035410-8ddd-4164-9baf-f35f9781322d'})


In [7]:
# Author IN [Alice, Bob]
results = await db.metadata_filter(
    filter={"author": {"$in": ["Alice", "Bob"]}},
    k=5,
)
print(f"📋 Author in [Alice, Bob]: {len(results)} results")
for r in results:
    print(f"  {r['metadata']['author']}: {r['content'][:60]}...")

📋 Author in [Alice, Bob]: 5 results
  Alice: PostgreSQL 17 introduces incremental backup and improved COP...
  Bob: pgvector 0.8 adds iterative scan and binary quantization sup...
  Bob: Kubernetes operators automate complex stateful application m...
  Alice: DiskANN enables billion-scale vector search with memory opti...
  Alice: RAG pipelines combine retrieval with LLM generation for grou...


In [8]:
# Count matching docs
count = await db.count_by_metadata(filter={"topic": "AI"})
print(f"📊 AI documents: {count}")

📊 AI documents: 4


## 3. Ensemble Search (Metadata + Hybrid)

In [9]:
results = await (
    db.query("vector search database")
    .hybrid()
    .fts()
    .where({"year": {"$gte": 2023}})
    .weights(semantic=0.6, keyword=0.4)
    .limit(3)
    .to_list()
)

print("🎼 Hybrid (year≥2023, semantic+keyword):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:70]}...")

🎼 Hybrid (year≥2023, semantic+keyword):
  1. [0.6000] DiskANN enables billion-scale vector search with memory optimization....
  2. [0.4068] pgvector 0.8 adds iterative scan and binary quantization support....
  3. [0.0903] Connection pooling with PgBouncer improves PostgreSQL scalability....


## 4. Universal Keyword Search (Content + Metadata)

In [10]:
results = await (
    db.query("Alice").keyword().fts().universal(metadata_fields=["author"]).limit(5).to_list()
)

print("🔤 Universal keyword 'Alice' (content + metadata):")
for r in results:
    print(f"  [{r['score']:.4f}] {r['content'][:60]}... (author: {r['metadata'].get('author')})")

🔤 Universal keyword 'Alice' (content + metadata):
  [0.0000] PostgreSQL 17 introduces incremental backup and improved COP... (author: Alice)
  [0.0000] DiskANN enables billion-scale vector search with memory opti... (author: Alice)
  [0.0000] RAG pipelines combine retrieval with LLM generation for grou... (author: Alice)


## 5. Recall Measurement

In [11]:
recall = await db.compute_recall(
    test_queries=["vector search", "database indexing", "machine learning"],
    k=5,
)
print("📏 Recall Results:")
for k, v in recall.items():
    print(f"  {k}: {v}")

📏 Recall Results:
  recall@k: 1.0
  queries_tested: 3
  k: 5


## 6. Query Plan Analysis

In [12]:
plan = await db.explain_query("vector search", search_method="semantic_search", k=3)
print("📋 Query Plan:")
for line in plan:
    print(f"  {line}")

📋 Query Plan:
  Limit  (cost=3.25..3.26 rows=3 width=88) (actual time=0.082..0.087 rows=3 loops=1)
    Output: langchain_id, content, langchain_metadata, ((embedding <=> '[-0.020554194,0.028560294,-0.023710858,-0.04844301,0.041288126,-0.013491078,0.015439656,-0.069064535,-0.034856796,-0.019757472,0.022188302,0.045352295,0.021023665,0.0155901965,-0.07663318,-0.0008396352,-0.05441372,0.08402941,0.07154116,-0.0068481327,-0.03340588,0.0126015,-0.025495453,-0.036164157,0.031640783,0.079223216,0.08075506,-0.063268624,0.043980267,-0.06778013,0.07776129,0.0333874,0.033928346,0.102353185,-0.13224757,-0.016929178,-0.10489835,0.0447305,-0.04854606,0.0602356,-0.055292983,0.04415148,0.0061615915,0.047821432,0.08498168,0.078164294,-0.05636514,0.009237063,0.09502853,-0.053061403,-0.10585932,-0.0739221,-0.070041314,-0.00027330566,-0.035571992,-0.026664907,0.03389673,-0.051015288,0.06457585,-0.04101822,0.061936792,-0.07968409,0.03531671,-0.02972459,0.070339724,-0.013861317,0.025630888,0.046659533,0.014

## 7. Collection Health Check

In [13]:
health = await db.validate_collection()
print(f"🏥 Collection healthy: {health.get('is_healthy', 'N/A')}")
if health.get("issues"):
    for issue in health["issues"]:
        print(f"  ⚠ {issue}")

🏥 Collection healthy: N/A


## 8. Export & Import

In [14]:
# Export to JSON
await db.export_to_json("backup_advanced.json", include_embeddings=False)
print("💾 Exported to backup_advanced.json")

# Check file size
size = os.path.getsize("backup_advanced.json")
print(f"   File size: {size / 1024:.1f} KB")

💾 Exported to backup_advanced.json
   File size: 2.9 KB


In [15]:
await db.delete_table()
await db.close()
print("🧹 Cleaned up")

🧹 Cleaned up
